# OpenPOM (MPNN-POM) on the Aromacle scaffold split

Run on a **Colab GPU runtime** (Runtime > Change runtime type > T4 GPU).

Colab's default kernel is Python 3.13, but OpenPOM's stack is pinned to **Python 3.10**
(the only version with wheels for torch 2.4.0 / CUDA 12.4 / DGL). So this notebook builds an
**isolated Python 3.10 environment with `uv`** and installs everything there.

The notebook kernel itself only uses the standard library + pandas; it never imports
`torch` / `dgl` / `deepchem`. The sanity check and the training both run as subprocesses
through `/content/pom_env/bin/python`.

Pins (from the current OpenPOM README): Python 3.10, `torch==2.4.0`, CUDA 12.4,
DGL from `data.dgl.ai/wheels/torch-2.4/cu124`, DeepChem nightly, `openpom`.

Inputs are read from `/content/drive/MyDrive/Aromacle`; results are copied back there.

In [ ]:
# Confirm a GPU is attached (expect a Tesla T4 table).
!nvidia-smi

In [ ]:
# Mount Google Drive (inputs live in MyDrive/Aromacle).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_DIR = '/content/drive/MyDrive/Aromacle'
REPO_DIR = '/content/Aromacle'
VENV_DIR = '/content/pom_env'
PYTHON = VENV_DIR + '/bin/python'

print(sorted(os.listdir(DRIVE_DIR)))

## 1. Build an isolated Python 3.10 environment

Versions are taken **exactly** from the current OpenPOM README. `numpy<2` is added because the
DGL 2.4 / CUDA 12.4 and RDKit wheels for this stack are built against NumPy 1.x and fail to
import under NumPy 2.

In [ ]:
# uv can provision a standalone CPython 3.10 even though the kernel is 3.13.
!pip install -q uv
!uv venv --python 3.10 /content/pom_env

# torch 2.4.0 + CUDA 12.4, exactly as pinned by the OpenPOM README.
!uv pip install --python /content/pom_env/bin/python torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
# DeepChem nightly (pip --pre  ==  uv --prerelease allow).
!uv pip install --python /content/pom_env/bin/python "numpy<2" --prerelease allow deepchem

# DGL matching torch-2.4 / cu124, from DGL's own wheel index.
!uv pip install --python /content/pom_env/bin/python "numpy<2" "dgl==2.4.0" --find-links https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html

# OpenPOM (its deps include dgllife / pyrfume). Keep dgl pinned to the cu124 build.
!uv pip install --python /content/pom_env/bin/python "numpy<2" "dgl==2.4.0" openpom --find-links https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html

In [ ]:
# Sanity check runs in the venv as a SUBPROCESS (the kernel must not import torch/dgl).
import subprocess

SANITY = '''
import sys, torch, dgl, deepchem
print('python        :', sys.version.split()[0])
print('torch         :', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda runtime  :', torch.version.cuda)
print('dgl           :', dgl.__version__)
print('deepchem      :', deepchem.__version__)
'''
subprocess.run([PYTHON, '-c', SANITY], check=True)

## 2. Get the Aromacle code

Clone the repository and install it **into the 3.10 venv** (editable). If the repository is
private, clone with a token or upload `src/` + `pyproject.toml` to `/content/Aromacle`.

In [ ]:
!git clone --depth 1 https://github.com/samragyi22/Aromacle.git /content/Aromacle || echo 'repo already present'
!uv pip install --python /content/pom_env/bin/python -e /content/Aromacle

In [ ]:
CSV_PATH = os.path.join(DRIVE_DIR, 'curated_GS_LF_merged_4983.csv')
SPLIT_PATH = os.path.join(DRIVE_DIR, 'split_seed42.json')
RESULTS_DIR = os.path.join(REPO_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(CSV_PATH), CSV_PATH
assert os.path.exists(SPLIT_PATH), SPLIT_PATH
print('csv    :', CSV_PATH)
print('split  :', SPLIT_PATH)
print('results:', RESULTS_DIR)

## 3. Train MPNN-POM on the Aromacle split

Trains on the train indices, early-stops on the val split (patience on mean val ROC-AUC),
restores the best checkpoint, and writes test-split probabilities.

In [ ]:
import subprocess

cmd = [
    PYTHON, '-m', 'aromacle.gnn.train_openpom',
    '--csv', CSV_PATH,
    '--split', SPLIT_PATH,
    '--results-dir', RESULTS_DIR,
    '--epochs', '40',
    '--patience', '5',
    '--batch-size', '128',
    '--seed', '42',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True, cwd=REPO_DIR)

In [ ]:
import json
import pandas as pd

preds = pd.read_csv(os.path.join(RESULTS_DIR, 'openpom_test_preds.csv'))
print('predictions:', preds.shape)  # (498, 138)
display(preds.head())

config = json.load(open(os.path.join(RESULTS_DIR, 'openpom_config.json')))
print('best epoch:', config['best_epoch'], '| val ROC-AUC:', round(config['best_val_roc_auc'], 4))
if config['history']:
    print('last epoch:', config['history'][-1])

In [ ]:
import shutil

for name in ('openpom_test_preds.csv', 'openpom_config.json'):
    shutil.copy(os.path.join(RESULTS_DIR, name), os.path.join(DRIVE_DIR, name))
print('copied results to', DRIVE_DIR)

## 4. Next: evaluate locally on identical terms

Download `openpom_test_preds.csv` from Drive into the repo's `results/`, then run:

```bash
python -m aromacle.evaluate_preds --preds results/openpom_test_preds.csv --model openpom --features mpnn
```

This uses the same evaluation logic as the classical baselines (same skipped labels, ROC-AUC,
PR-AUC, 500-resample bootstrap CIs) and upserts the OpenPOM row into `results/baseline_metrics.csv`.

**Report honestly:** only claim OpenPOM "beats" a baseline if the bootstrap CIs support it.